# Business Entity Matching Pipeline — Real Data (Steps 1–4)

This notebook adapts the sample matching pipeline to the real `source1.tsv`,
`source2.tsv`, `source3.tsv`, and `ground_truth.tsv` files.

**Known data caveat:** the files currently loaded are row-subsets copied out of
larger files, so IDs across `ground_truth.tsv` and `source1/2/3.tsv` will not
resolve against each other yet. Steps 1–4 (inspection, normalization, blocking,
feature engineering) work regardless. Steps 5+ (labeling, training, threshold
tuning) require the full, ID-aligned files and will be added once those are
available.

Pipeline covered here:

```
Raw Data → Inspection → Normalization → Blocking → Similarity Features
```


## Step 1: Understand and Inspect the Real Data

**Objective:** Load the three sources and ground truth, confirm column
mapping, and quantify what matters for later steps — nulls, duplicates,
country mix, script mix — without assuming the sample dataset's structure.


In [10]:
print("hello world")

hello world


In [17]:
# ============================================================
# STEP 1: LOAD AND INSPECT REAL DATA
# ============================================================
import pandas as pd
import re
from IPython.display import display

pd.set_option("display.max_colwidth", 100)

# ---- Configurable paths ----
PATH_SOURCE1 = r"dataset\train\train_source1.tsv"
PATH_SOURCE2 = r"dataset\train\train_source2.tsv"
PATH_SOURCE3 = r"dataset\train\train_source3.tsv"
PATH_GROUND_TRUTH = r"dataset\train\train_ground_truth.tsv"

source1 = pd.read_csv(PATH_SOURCE1, sep="\t")
source2 = pd.read_csv(PATH_SOURCE2, sep="\t")
source3 = pd.read_csv(PATH_SOURCE3, sep="\t")
ground_truth = pd.read_csv(PATH_GROUND_TRUTH, sep="\t")

print("Shapes:")
print("  source1:", source1.shape)
print("  source2:", source2.shape)
print("  source3:", source3.shape)
print("  ground_truth:", ground_truth.shape)

print("\nColumns:")
print("  source1:", list(source1.columns))
print("  source2:", list(source2.columns))
print("  source3:", list(source3.columns))
print("  ground_truth:", list(ground_truth.columns))

Shapes:
  source1: (2206821, 4)
  source2: (5034616, 4)
  source3: (5285603, 4)
  ground_truth: (2206821, 2)

Columns:
  source1: ['entity_id', 'business_name', 'business_address', 'country']
  source2: ['entity_id', 'business_name', 'business_address', 'country']
  source3: ['entity_id', 'business_name', 'business_address', 'country']
  ground_truth: ['source1_entity_id', 'matched_entity_ids']


In [18]:
# ---- Nulls ----
for name, df in [("source1", source1), ("source2", source2),
                  ("source3", source3), ("ground_truth", ground_truth)]:
    print(f"\n--- {name} null counts ---")
    print(df.isnull().sum())



--- source1 null counts ---
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

--- source2 null counts ---
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

--- source3 null counts ---
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

--- ground_truth null counts ---
source1_entity_id          0
matched_entity_ids    123247
dtype: int64


In [19]:
# ---- Uniqueness of IDs ----
for name, df, id_col in [("source1", source1, "entity_id"),
                          ("source2", source2, "entity_id"),
                          ("source3", source3, "entity_id")]:
    print(f"{name}: {df[id_col].nunique()} unique / {len(df)} rows, "
          f"duplicates={df[id_col].duplicated().sum()}")


source1: 2206821 unique / 2206821 rows, duplicates=0
source2: 5034616 unique / 5034616 rows, duplicates=0
source3: 5285603 unique / 5285603 rows, duplicates=0


In [20]:
# ---- Country distribution ----
print("source1 countries:\n", source1["country"].value_counts())
print("\nsource2 countries:\n", source2["country"].value_counts())
print("\nsource3 countries:\n", source3["country"].value_counts())


source1 countries:
 country
US       1323633
India     883188
Name: count, dtype: int64

source2 countries:
 country
US       3016817
India    2017799
Name: count, dtype: int64

source3 countries:
 country
US       3170056
India    2115547
Name: count, dtype: int64


In [21]:
# ---- Script / language detection ----
# Business data here mixes English (Latin script) and Hindi (Devanagari script).
# We detect this directly rather than assuming English-only text.

def detect_script(text):
    if not isinstance(text, str) or text.strip() == "":
        return "empty"
    if re.search(r"[\u0900-\u097F]", text):
        return "devanagari"
    if re.search(r"[A-Za-z]", text):
        return "latin"
    return "other"

for name, df in [("source1", source1), ("source2", source2), ("source3", source3)]:
    df["_name_script"] = df["business_name"].apply(detect_script)
    print(f"\n{name} business_name script distribution:")
    print(df["_name_script"].value_counts())



source1 business_name script distribution:
_name_script
latin    2206821
Name: count, dtype: int64

source2 business_name script distribution:
_name_script
latin         4568099
devanagari     269424
other          197091
empty               2
Name: count, dtype: int64

source3 business_name script distribution:
_name_script
latin         5021720
devanagari     158003
other          105867
empty              13
Name: count, dtype: int64


In [22]:
# ---- Ground truth structure check ----
# Checks whether ground-truth IDs resolve against the loaded source files.
# With partial/sampled files, this will show 0% overlap -- expected until the
# FULL source1/2/3 files are loaded together with the FULL ground truth.

def expand_matches(val):
    if pd.isna(val):
        return []
    return [x.strip() for x in val.split(",") if x.strip()]

ground_truth["_matched_list"] = ground_truth["matched_entity_ids"].apply(expand_matches)

gt_s1_ids = set(ground_truth["source1_entity_id"])
gt_matched_ids = set(x for lst in ground_truth["_matched_list"] for x in lst)
gt_matched_s2 = {x for x in gt_matched_ids if x.startswith("S2-")}
gt_matched_s3 = {x for x in gt_matched_ids if x.startswith("S3-")}

overlap_s1 = gt_s1_ids & set(source1["entity_id"])
overlap_s2 = gt_matched_s2 & set(source2["entity_id"])
overlap_s3 = gt_matched_s3 & set(source3["entity_id"])

print(f"Ground truth S1 ids resolvable in source1: {len(overlap_s1)}/{len(gt_s1_ids)}")
print(f"Ground truth S2 ids resolvable in source2: {len(overlap_s2)}/{len(gt_matched_s2)}")
print(f"Ground truth S3 ids resolvable in source3: {len(overlap_s3)}/{len(gt_matched_s3)}")

if len(overlap_s1) == 0:
    print("\n⚠️  No overlap detected. If this is the FULL dataset, investigate immediately —"
          "\n    training/blocking-recall/threshold steps cannot run until this resolves.")


Ground truth S1 ids resolvable in source1: 2206821/2206821
Ground truth S2 ids resolvable in source2: 3693619/3693619
Ground truth S3 ids resolvable in source3: 3944746/3944746


**Validation check:** eyeball a few rows and confirm the Devanagari rows got tagged correctly.

In [23]:
display(source1.head())
display(source2.head())
display(source3.head())
display(ground_truth.head())
display(source2[source2["_name_script"] == "devanagari"][["entity_id", "business_name", "country"]])


,entity_id,business_name,business_address,country,_name_script
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,latin
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,latin
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,latin
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,latin
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",India,latin


,entity_id,business_name,business_address,country,_name_script
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India,devanagari
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,latin
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India,devanagari
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US,latin
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,latin


,entity_id,business_name,business_address,country,_name_script
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US,latin
1,S3-859268022,International South Consultants Private Ltd,NaN,India,latin
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US,latin
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US,latin
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block Jayanagar, Bengaluru Urban, Bangalore, ಕರ್ನಾಟಕ",India,latin


,source1_entity_id,matched_entity_ids,_matched_list
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364","[S2-681193310, S2-743505751, S3-775321672, S3-11291185, S3-860443364]"
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-384364074","[S2-249013014, S2-197070651, S3-478195123, S3-384364074]"
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467","[S2-790675320, S2-479876582, S3-878454467]"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215","[S2-153058913, S2-24659151, S3-679606215]"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785","[S2-478959098, S2-553508714, S2-625774905, S3-728090388, S3-928796641, S3-449308785]"


,entity_id,business_name,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,India
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,India
11,S2-615967906,सन कंस्ट्रक्शंस प्राइवेट लिमिटेड,India
16,S2-566688803,रियल मॉडर्न फूड लिमिटेड,India
60,S2-64510343,अल्फा अल टेक्नोलॉजीज प्राइवेट लिमिटेड,India
...,...,...,...
5034481,S2-859414675,गुरु इंफ्रा प्रा. लि.,India
5034530,S2-267739717,टेक सॉल्यूशंस प्राइवेट लिमिटेड,India
5034567,S2-592496211,माँ टेक्नोलॉजीज लिमिटेड,India
5034602,S2-664921582,इंटरनेशनल हेल्थकेयर प्राइवेट लिमिटेड,India


## Step 2: Normalization

**Objective:** Build normalized versions of name/address/country without
destroying the originals, and handle the Latin/Devanagari mix explicitly.

For the small number of Devanagari rows, we do **not** transliterate
(no external APIs, no blind translation per project rules). Instead we rely
on Unicode-safe cleaning here, plus **character n-gram TF-IDF** in Step 4,
which works across scripts without translation.


In [24]:
import unicodedata

ABBREVIATIONS = {
    r"\bpvt\b": "private",
    r"\bltd\b": "limited",
    r"\bltd\.\b": "limited",
    r"\binc\b": "incorporated",
    r"\bco\b": "company",
    r"\bcorp\b": "corporation",
    r"\bllp\b": "limited liability partnership",
    r"\bllc\b": "limited liability company",
    r"\bst\b": "street",
    r"\bst\.\b": "street",
    r"\brd\b": "road",
    r"\brd\.\b": "road",
    r"\bave\b": "avenue",
    r"\bave\.\b": "avenue",
    r"\bapt\b": "apartment",
}

def unicode_clean(text):
    """Unicode-safe normalization: NFKC form, keeps non-Latin scripts intact."""
    if not isinstance(text, str):
        return ""
    return unicodedata.normalize("NFKC", text)

def normalize_name(text):
    text = unicode_clean(text)
    text = text.lower()
    text = text.replace("&", " and ")
    # remove punctuation but keep unicode word characters (so Devanagari survives)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    for pattern, replacement in ABBREVIATIONS.items():
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_address(text):
    text = unicode_clean(text)
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    for pattern, replacement in ABBREVIATIONS.items():
        text = re.sub(pattern, replacement, text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_country(text):
    if not isinstance(text, str):
        return ""
    return text.strip().lower()

for df in [source1, source2, source3]:
    df["name_normalized"] = df["business_name"].apply(normalize_name)
    df["address_normalized"] = df["business_address"].fillna("").apply(normalize_address)
    df["country_normalized"] = df["country"].apply(normalize_country)


**Validation check:**

In [25]:
display(source1[["business_name", "name_normalized", "business_address", "address_normalized"]].sample(5, random_state=1))
display(source2[source2["_name_script"] == "devanagari"][["business_name", "name_normalized"]])


,business_name,name_normalized,business_address,address_normalized
1524360,Meek Principal,meek principal,"Fort Recovery, OH, 2620 Sawmill Road",fort recovery oh 2620 sawmill road
585589,Family Charities II,family charities ii,"2819 Sanctuary Cove, Katy, TX",2819 sanctuary cove katy tx
855245,Global Complete Vendome LLC,global complete vendome limited liability company,"3106 London Lane, Missouri City, TX",3106 london lane missouri city tx
1926943,Taylor Battery Holdings,taylor battery holdings,"3142 Village Pine Cove, West Valley City, UT",3142 village pine cove west valley city ut
2038564,Complete Premier Tbk,complete premier tbk,"25622 6, Nappanee, IN",25622 6 nappanee in


,business_name,name_normalized
0,राम मार्केटिंग प्राइवेट लिमिटेड,र म म र क ट ग प र इव ट ल म ट ड
2,आदित्य प्रॉपर्टीज एलएलपी,आद त य प र पर ट ज एलएलप
11,सन कंस्ट्रक्शंस प्राइवेट लिमिटेड,सन क स ट रक श स प र इव ट ल म ट ड
16,रियल मॉडर्न फूड लिमिटेड,र यल म डर न फ ड ल म ट ड
60,अल्फा अल टेक्नोलॉजीज प्राइवेट लिमिटेड,अल फ अल ट क न ल ज ज प र इव ट ल म ट ड
...,...,...
5034481,गुरु इंफ्रा प्रा. लि.,ग र इ फ र प र ल
5034530,टेक सॉल्यूशंस प्राइवेट लिमिटेड,ट क स ल य श स प र इव ट ल म ट ड
5034567,माँ टेक्नोलॉजीज लिमिटेड,म ट क न ल ज ज ल म ट ड
5034602,इंटरनेशनल हेल्थकेयर प्राइवेट लिमिटेड,इ टरन शनल ह ल थक यर प र इव ट ल म ट ड


## Step 3: Candidate Generation / Blocking

**Objective:** Generate candidate S1↔S2 and S1↔S3 pairs using multiple loose
rules (union), so we don't lose true matches to overly strict blocking.

Same-country + (name similarity OR address similarity) via RapidFuzz
`token_set_ratio`, like the sample pipeline — but thresholds are configurable
constants to retune once blocking recall is measurable against the full
ground truth.


In [26]:
%pip install rapidfuzz

  Using cached rapidfuzz-3.14.6-cp314-cp314-win_amd64.whl.metadata (12 kB)
Using cached rapidfuzz-3.14.6-cp314-cp314-win_amd64.whl (1.8 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
from rapidfuzz import fuzz

# Configurable thresholds -- retune once blocking recall can be measured
NAME_THRESHOLD = 60
ADDRESS_THRESHOLD = 60

def generate_candidates(s1_df, other_df, other_prefix):
    """
    Union of blocking rules:
      Rule A: same country + name similarity >= NAME_THRESHOLD
      Rule B: same country + address similarity >= ADDRESS_THRESHOLD
      Rule C: same country + (name AND address) both moderately similar
    Returns a DataFrame of candidate pairs with similarity scores attached.
    """
    candidates = []
    # group by country to avoid an O(n*m) full cross join across all countries
    for country_val, s1_group in s1_df.groupby("country_normalized"):
        other_group = other_df[other_df["country_normalized"] == country_val]
        if other_group.empty:
            continue
        for _, s1_row in s1_group.iterrows():
            for _, o_row in other_group.iterrows():
                name_sim = fuzz.token_set_ratio(s1_row["name_normalized"], o_row["name_normalized"])
                addr_sim = fuzz.token_set_ratio(s1_row["address_normalized"], o_row["address_normalized"])

                rule_a = name_sim >= NAME_THRESHOLD
                rule_b = addr_sim >= ADDRESS_THRESHOLD
                rule_c = (name_sim >= NAME_THRESHOLD - 15) and (addr_sim >= ADDRESS_THRESHOLD - 15)

                if rule_a or rule_b or rule_c:
                    candidates.append({
                        "s1_id": s1_row["entity_id"],
                        "candidate_id": o_row["entity_id"],
                        "candidate_source": other_prefix,
                        "block_name_sim": name_sim,
                        "block_address_sim": addr_sim,
                    })
    return pd.DataFrame(candidates)

candidates_s2 = generate_candidates(source1, source2, "S2")
candidates_s3 = generate_candidates(source1, source3, "S3")

candidate_pairs_raw = pd.concat([candidates_s2, candidates_s3], ignore_index=True)
print("Total candidate pairs generated:", len(candidate_pairs_raw))
print(candidate_pairs_raw["candidate_source"].value_counts())


NameError: name 'source1' is not defined

**Validation check — blocking recall** (run once full files/ground truth are aligned; currently expected to show `0/0`):

In [ ]:
def check_blocking_recall(ground_truth, candidate_pairs):
    total_true = 0
    found_true = 0
    candidate_set = set(zip(candidate_pairs["s1_id"], candidate_pairs["candidate_id"]))
    for _, row in ground_truth.iterrows():
        for match_id in row["_matched_list"]:
            total_true += 1
            if (row["source1_entity_id"], match_id) in candidate_set:
                found_true += 1
    recall = found_true / total_true if total_true else float("nan")
    print(f"Blocking recall: {found_true}/{total_true} = {recall:.3f}")
    return recall

check_blocking_recall(ground_truth, candidate_pairs_raw)


## Step 4: Similarity Feature Engineering

**Objective:** Turn each candidate pair into numeric features for the ML
model, adding a **character-level TF-IDF** feature specifically to handle
the Devanagari rows (word-level tokenization would treat those as one giant
out-of-vocabulary token against Latin text).

TF-IDF vectorizers are fit once on the full corpus across all three sources
(not per-pair). Both `analyzer="word"` (for the mostly-English content) and
`analyzer="char_wb"` (2–4 character n-grams, robust across scripts) are used.


In [ ]:
%pip install rapidfuzz scikit-learn

In [ ]:
import numpy as np
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---- Fit vectorizers once on the combined corpus ----
all_names = pd.concat([source1["name_normalized"], source2["name_normalized"], source3["name_normalized"]])
all_addresses = pd.concat([source1["address_normalized"], source2["address_normalized"], source3["address_normalized"]])

name_word_vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))
name_char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
addr_word_vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2))

name_word_vectorizer.fit(all_names)
name_char_vectorizer.fit(all_names)
addr_word_vectorizer.fit(all_addresses)

# ---- Lookup dicts: entity_id -> row, for fast feature computation ----
def build_lookup(df):
    return df.set_index("entity_id").to_dict(orient="index")

lookup_s1 = build_lookup(source1)
lookup_s2 = build_lookup(source2)
lookup_s3 = build_lookup(source3)

def get_row(entity_id, source_prefix):
    if source_prefix == "S2":
        return lookup_s2[entity_id]
    elif source_prefix == "S3":
        return lookup_s3[entity_id]
    else:
        raise ValueError(source_prefix)

def jaccard_token_sim(a, b):
    set_a, set_b = set(a.split()), set(b.split())
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


In [ ]:
def build_features(candidate_pairs):
    rows = []
    # batch-transform for TF-IDF to avoid per-row fit/transform overhead
    s1_names = [lookup_s1[i]["name_normalized"] for i in candidate_pairs["s1_id"]]
    cand_names = [get_row(cid, src)["name_normalized"]
                  for cid, src in zip(candidate_pairs["candidate_id"], candidate_pairs["candidate_source"])]
    s1_addrs = [lookup_s1[i]["address_normalized"] for i in candidate_pairs["s1_id"]]
    cand_addrs = [get_row(cid, src)["address_normalized"]
                  for cid, src in zip(candidate_pairs["candidate_id"], candidate_pairs["candidate_source"])]

    s1_name_word_vec = name_word_vectorizer.transform(s1_names)
    cand_name_word_vec = name_word_vectorizer.transform(cand_names)
    s1_name_char_vec = name_char_vectorizer.transform(s1_names)
    cand_name_char_vec = name_char_vectorizer.transform(cand_names)
    s1_addr_vec = addr_word_vectorizer.transform(s1_addrs)
    cand_addr_vec = addr_word_vectorizer.transform(cand_addrs)

    name_word_cos = np.array([cosine_similarity(s1_name_word_vec[i], cand_name_word_vec[i])[0, 0]
                               for i in range(len(candidate_pairs))])
    name_char_cos = np.array([cosine_similarity(s1_name_char_vec[i], cand_name_char_vec[i])[0, 0]
                               for i in range(len(candidate_pairs))])
    addr_cos = np.array([cosine_similarity(s1_addr_vec[i], cand_addr_vec[i])[0, 0]
                          for i in range(len(candidate_pairs))])

    for idx, (_, pair) in enumerate(candidate_pairs.iterrows()):
        s1_name, cand_name = s1_names[idx], cand_names[idx]
        s1_addr, cand_addr = s1_addrs[idx], cand_addrs[idx]

        rows.append({
            "s1_id": pair["s1_id"],
            "candidate_id": pair["candidate_id"],
            "candidate_source": pair["candidate_source"],

            # name features
            "name_exact_match": int(s1_name == cand_name),
            "name_rapidfuzz": fuzz.token_set_ratio(s1_name, cand_name) / 100.0,
            "name_levenshtein": Levenshtein.normalized_similarity(s1_name, cand_name),
            "name_jaccard": jaccard_token_sim(s1_name, cand_name),
            "name_tfidf_word_cos": name_word_cos[idx],
            "name_tfidf_char_cos": name_char_cos[idx],
            "name_token_overlap": len(set(s1_name.split()) & set(cand_name.split())),
            "name_len_diff": abs(len(s1_name) - len(cand_name)),

            # address features
            "addr_exact_match": int(s1_addr == cand_addr),
            "addr_rapidfuzz": fuzz.token_set_ratio(s1_addr, cand_addr) / 100.0,
            "addr_levenshtein": Levenshtein.normalized_similarity(s1_addr, cand_addr),
            "addr_jaccard": jaccard_token_sim(s1_addr, cand_addr),
            "addr_tfidf_cos": addr_cos[idx],
            "addr_token_overlap": len(set(s1_addr.split()) & set(cand_addr.split())),
            "addr_len_diff": abs(len(s1_addr) - len(cand_addr)),

            # other
            "country_match": int(lookup_s1[pair["s1_id"]]["country_normalized"] ==
                                  get_row(pair["candidate_id"], pair["candidate_source"])["country_normalized"]),
        })
    return pd.DataFrame(rows)

candidate_features = build_features(candidate_pairs_raw)
print(candidate_features.shape)
candidate_features.head()


**Validation check:**

In [ ]:
# Sanity: exact-name-match pairs should score high on rapidfuzz/levenshtein too
display(candidate_features[candidate_features["name_exact_match"] == 1].head())

# Sanity: features should be within expected ranges
candidate_features.describe()


## Next Steps (Step 5 onward)

Once the full, ID-aligned `source1.tsv`, `source2.tsv`, `source3.tsv`, and
`ground_truth.tsv` are available:

1. Re-run the **blocking recall** check in Step 3 — retune `NAME_THRESHOLD`
   / `ADDRESS_THRESHOLD` based on the real recall number.
2. **Step 5:** label `candidate_features` using `ground_truth`, split by
   `s1_id` with `GroupShuffleSplit`, train an `XGBClassifier`.
3. **Step 6:** tune the decision threshold on validation data using F0.5
   (`sklearn.metrics.fbeta_score`, `beta=0.5`), not the default 0.5 cutoff.
4. **Step 7:** apply the finalized pipeline to test data and produce
   `matching_results.tsv` and `candidate_pairs.tsv`.
